In [ ]:
import geopandas as gpd
import matplotlib as mpl
import matplotlib.patheffects as mpe
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from project.utils.data import enrich, load_hillshade
from project.utils.metrics import compute_metrics
from project.utils.paths import get_path
from project.utils.plotting import ProjectCustomExtent, despine
from project.utils.unit_conversion import convert_xr_m3s_to_mmd

plt.rcParams["figure.dpi"] = 150

# Switzerland LV95 with extended limits to cover 'Hydrological Switzerland'.
lv95 = ProjectCustomExtent(2056, [2474273, 2842000, 1058000, 1325000])

In [ ]:
# Load predictions.
ds = xr.open_zarr(get_path('predictions_file')).sel(
    time=slice('1993', None))

# Load HRU polygons.
hrus = gpd.read_file(
    get_path('hru_file'), use_arrow=True).set_index('EZGNR')

metrics_max = ['kge', 'nse', 'nnse', 'r']
metrics_min = ['rmse', 'bias2', 'varerr', 'phaseerr', 'relbias', 'flv', 'fhv']

metric_nice_names = {
    'bias2': 'Bias²',
    'varerr': 'Variance error',
    'phaseerr': 'Phase error',
    'nse': 'Nash-Sutcliffe Efficiency',
    'nnse': 'Normalized Nash-Sutcliffe Efficiency',
    'kge': 'Kling-Gupta Efficiency',
    'r': 'Pearson\'s r',
    'relbias': '%Bias',
    'flv': '%Low Flow Bias (<Q5)',
    'fhv': '%High Flow Bias (>Q98)',
}

# Compute metrics and add them to dataset.
met = compute_metrics(ds.Q_obs_m3s, ds.Q_pred_m3s, metrics=metrics_min + metrics_max, dim='time')
ds = xr.merge([ds, met])

for m in metrics_min + metrics_max:
    name = f'{m}_impr'

    # Create the target variable once per metric.
    # Use the metric itself as a template (and make sure it's float so NaN works).
    if name not in ds:
        ds[name] = xr.full_like(ds[m], np.nan, dtype=float).load()

    for model in ds.model.values:
        # Pair model with the corresponding other model
        model_type, split_type = model.split('_')
        other_model_type = 'routed' if model_type == 'lumped' else 'lumped'
        other_model = f'{other_model_type}_{split_type}'
        if other_model not in ds.model.values:
            continue

        value_model = ds.sel(model=model)[m]
        value_other = ds.sel(model=other_model)[m]

        if m in metrics_min:
            value_model = np.abs(value_model)
            value_other = np.abs(value_other)

        if m in metrics_min:
            # Lower is better
            pec_impr = (value_other - value_model) / (value_other + 1e-6) * 100.0
        else:
            # Higher is better
            pec_impr = (value_model - value_other) / (1 - value_other) * 100.0

        # Write only the slice for this model
        ds[name].loc[dict(model=model)] = pec_impr


ds['R_pred_mean'] = ds['R_pred'].mean('time').compute()

In [ ]:
rivers = gpd.read_file(get_path('raw_data_root') / 'raw/rivernet', use_arrow=True)

In [ ]:
bg = load_hillshade(
    hillshade_dir=get_path('raw_data_root') / 'processed',
    hrus=hrus,
    ds=ds,
    dem_path=get_path('raw_data_root') / 'raw/merit.zarr',
    use_terrain=False,
    target_crs='EPSG:2056',
    force_recompute=False)

## Case study

In [ ]:
def plpt_ts(ds, ax, time_start, time_end, pred_labels):
    ds_s = ds.sel(time=slice(time_start, time_end))
    ds_s.Q_obs_m3s.sel(
        time=slice(time_start, time_end)).plot(
            ax=ax,
            color='0.0',
            lw=1.0,
            label='Observed'
        )

    for i, model in enumerate(['routed_random', 'lumped_random']):
        color = ["#DAA507", "#1E88E5"][i]

        ds_s.Q_pred_m3s.sel(model=model).plot(
            ax=ax,
            label=pred_labels[i],
            alpha=1.0,
            lw=1.2,
            ls='--',
            dashes=[3, 1],
            color=color,
            zorder=5-i
        )


fig = plt.figure(figsize=(10, 5))

gs = mpl.gridspec.GridSpec(2, 2, figure=fig, wspace=0.1, hspace=0.25)

ds_aare = ds.sel(hru=145830)

ax_tl = fig.add_subplot(gs[0, 0])
ax_tr = fig.add_subplot(gs[0, 1], sharey=ax_tl)
ax_b = fig.add_subplot(gs[1, :])

# z0 = ('2020-01-01', '2022-12-31')
# z1 = ('2020-05-01', '2020-12-31')
# z2 = ('2021-05-01', '2021-12-31')

z0 = ('2006-01-01', '2008-12-31')
z1 = ('2006-05-01', '2006-12-31')
z2 = ('2007-05-01', '2007-12-31')

routed_random_kge = ds_aare.kge.sel(model='routed_random').item()
lumped_random_kge = ds_aare.kge.sel(model='lumped_random').item()

labels = [
    f'DRAIN (distributed; KGE={routed_random_kge:.2f})',
    f'LSTM (lumped; KGE={lumped_random_kge:.2f})'
]

plpt_ts(ds_aare, ax_b, *z0, labels)
plpt_ts(ds_aare, ax_tl, *z1, labels)
plpt_ts(ds_aare, ax_tr, *z2, labels)

# for i, model in enumerate(['routed_random', 'lumped_random']):
#     color = ['tab:red', 'tab:blue'][i]
#     Q_pred_m3s.sel(model=model).plot(ax=ax_b, label=model.replace('_', ' ').title(), lw=0.7, ls='--', alpha=0.8, color=color)

# Add the connection patches

for s, (zoom_start, zoom_end) in enumerate([z1, z2]):
    top_axis = [ax_tl, ax_tr][s]

    zoom_start = pd.Timestamp(zoom_start)
    zoom_end = pd.Timestamp(zoom_end)

    ax_b.axvspan(zoom_start, zoom_end, ec='k', fc='0.92', lw=0.4, ls='--')

    for z, zoom in enumerate([zoom_start, zoom_end]):

        fig.add_artist(mpl.patches.ConnectionPatch(
            xyA=(z, 0), coordsA=top_axis.transAxes,
            xyB=(mpl.dates.date2num(zoom), ax_b.get_ylim()[1]), coordsB=ax_b.transData,
            color='k',
            lw=0.4,
            ls='--',
            zorder=0,
        ))

for i, ax in enumerate([ax_tl, ax_tr, ax_b]):
    ax.set_xlabel('')
    despine(ax)
    ax.set_title('')

    if i != 1:
        ax.set_ylabel('Discharge (m$^3$ s$^{-1}$)')
    else:
        ax.set_ylabel('')

ax_b.legend(loc='upper right', bbox_to_anchor=(1, 1), frameon=False)

for i, ax in enumerate([ax_tr, ax_b]):
    event = pd.Timestamp('2007-08-09')
    if i == 0:
        post_event = pd.Timestamp('2007-08-14')
    else:
        post_event = pd.Timestamp('2007-08-19')

    ax.axvline(mpl.dates.date2num(event), color='k', lw=0.6, ls='--')
    ax.text(
        mpl.dates.date2num(post_event),
        ax.get_ylim()[1] * 0.9,
        'Peak flood event',
        color='k',
        fontsize=6,
        rotation=90,
        va='top',
        ha='left',
        path_effects=[mpe.withStroke(linewidth=1, foreground="w")]
    )

fig.align_ylabels([ax_tl, ax_b])

fig.savefig('./figures/time_series.png', dpi=300, bbox_inches='tight')

# Contribution analysis

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

In [ ]:
hru = 145830

In [ ]:
def select_dayofyear_window(ds: xr.Dataset | xr.DataArray, date: str, window: int) -> xr.Dataset | xr.DataArray:
    center_doy = ds.sel(time=date).time.dt.dayofyear.item()
    query_days = np.arange(center_doy - window, center_doy + window + 1)
    return ds.sel(time=ds.time.dt.dayofyear.isin(query_days))

dss = xr.open_zarr(get_path('log_root') / 'routed/random/tune/source_contrib_145830.zarr')

In [ ]:
source_ref = select_dayofyear_window(dss.q_source_m3s, date='2007-08-09', window=10).mean('time').compute()
source_day = dss.q_source_m3s.sel(time='2007-08-09').compute()
source_diff = source_day - source_ref

In [ ]:
source = xr.Dataset({
    'ref_m3s': source_ref,
    'day_m3s': source_day,
    'diff_m3s': source_diff,
    'area': dss.area,
})
convert_xr_m3s_to_mmd(source, variables=['ref_m3s', 'day_m3s', 'diff_m3s']).compute()

In [ ]:
hrus_source = enrich(
    source.day_mmd.sel(lag=0), hrus.loc[source.hru], variables='lag_instant')
hrus_source = enrich(
    source.day_mmd.sel(lag=slice(3, 1)).sum('lag'), hrus_source, variables='lag_1w')
hrus_source = enrich(
    source.day_mmd.sel(lag=slice(30, 4)).sum('lag'), hrus_source, variables='lag_remaining')

hrus_source = enrich(
    ds.prec.sel(time='2007-08-09', hru=hrus_source.index), hrus_source, variables='prec_instant')
hrus_source = enrich(
    ds.prec.sel(time=slice('2007-08-06', '2007-08-08'), hru=hrus_source.index).mean('time'), hrus_source, variables='prec_1w')
hrus_source = enrich(
    ds.prec.sel(time=slice('2007-07-10', '2007-08-05'), hru=hrus_source.index).mean('time'), hrus_source, variables='prec_remaining')

hrus_source = enrich(
    ds.q_pred_mmd.sel(time='2007-08-09', model='routed_random', hru=hrus_source.index), hrus_source, variables='q_instant')
hrus_source = enrich(
    ds.q_pred_mmd.sel(time=slice('2007-08-06', '2007-08-08'), model='routed_random', hru=hrus_source.index).mean('time'), hrus_source, variables='q_1w')
hrus_source = enrich(
    ds.q_pred_mmd.sel(time=slice('2007-07-10', '2007-08-05'), model='routed_random', hru=hrus_source.index).mean('time'), hrus_source, variables='q_remaining')

hrus_source = enrich(source.area, hrus_source, variables='area')

In [ ]:
variables = [
    ['prec_instant', 'prec_1w', 'prec_remaining'],
    ['q_instant', 'q_1w', 'q_remaining'],
    ['lag_instant', 'lag_1w', 'lag_remaining'],
]
titles = [
    'August 9 (instant)',
    'August 6-8 (lag 1 to 3 days)',
    'July 10 to August 5 (lag 4 to 31 days)',
]
ylabels = [
    'Observed\nprecipitation (mean)',
    'Simulated\nrunoff (mean)',
    'Contribution to\nAugust 9 2007',
]

In [ ]:
from project.utils.unit_conversion import convert_mmd_to_m3s
from matplotlib.gridspec import GridSpec

In [ ]:
xmin, ymin, xmax, ymax = hrus_source.total_bounds
xrng = xmax - xmin
xmin -= xrng * 0.05
xmax += xrng * 0.05
yrng = ymax - ymin
ymin -= yrng * 0.05
ymax += yrng * 0.05
bounds = [xmin, xmax, ymin, ymax]

bg_bounds = bg.sel(lon=slice(*bounds[:2]), lat=slice(bounds[-1], bounds[-2])).load()

In [ ]:
fig = plt.figure(layout='constrained', figsize=(8, 4))

gs = GridSpec(3, 4, figure=fig, wspace=0.02, hspace=0.0, width_ratios=[1, 1, 1, 0.02])




for r in range(3):
    vmax = np.quantile(hrus_source[variables[r]], 0.98)
    norm = mpl.colors.Normalize(vmin=0, vmax=vmax)

    kwargs = {
        'cmap': ['GnBu', 'PuRd', 'BuPu'][r],
        'ec': 'k',
        'lw': 0.1,
        'alpha': 0.7,
        'legend': False,
    }

    for c in range(3):

        ax = fig.add_subplot(gs[r, c], projection=lv95)

        x = hrus_source.loc[hru].geometry.centroid.x
        y = hrus_source.loc[hru].geometry.centroid.y

        ax.set_extent(bounds, crs=lv95)
        # bg_bounds.plot.imshow(ax=ax, interpolation='bilinear', alpha=0.8, zorder=0)

        rivers.loc[rivers.CUM_LEN > 20000].plot(ax=ax, color='k', lw=0.5, zorder=20, alpha=0.7)

        var = variables[r][c]

        hrus_source.plot(
            column=var,
            ax=ax,
            norm=norm,
            **kwargs
        )

        if c == 0:
            ax.text(-0.1, 0.5, ylabels[r], rotation=90, va='center', ha='center', transform=ax.transAxes, fontsize=8)
        
        if r == 0:
            ax.set_title(titles[c], fontsize=8)
        else:
            ax.set_title('')

        if r == 2:
            contr = convert_mmd_to_m3s(hrus_source[var].values, hrus_source['area'].values).sum()
            ax.text(
                0.015, 0.98, f'Total contribution:\n{contr:.0f} m$^3$ s$^{{-1}}$', transform=ax.transAxes,
                ha='left', va='top', fontsize=7, bbox=dict(facecolor='0.93', alpha=1, edgecolor='none', pad=1), zorder=100)

        ax.scatter(x, y, ec='k', fc='tab:red', s=12, lw=0.6, zorder=9999)

    cbar_ax = fig.add_subplot(gs[r, -1])
    cax = cbar_ax.inset_axes([0.0, 0.1, 1.0, 0.8])
    mappable = mpl.cm.ScalarMappable(norm=norm, cmap=kwargs['cmap'])
    cbar = plt.colorbar(
        mappable,
        cax=cax,
        orientation='vertical',
        extend='max',
        label=['Precipitation (mm d$^{-1}$)', 'Runoff (mm d$^{-1}$)', 'Contribution (mm d$^{-1}$)'][r],
    )
    cbar.ax.yaxis.label.set_size(8)
    cbar.ax.tick_params(labelsize=8)
    cbar_ax.axis('off')

fig.savefig('./figures/case_study.png', dpi=300, bbox_inches='tight')